# Testing the Pydantic classes in execution
Init: Week of 14 Nov

Status: Complete and Debugged. Needs fallback methods + traceability

In [1]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from typing import List, Optional, Literal, Type
import json

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [2]:
# Defining the pydantic parsing structure

class Topic(BaseModel):
    """Represents the explicitly stated topic(s) of the document."""
    main_topic: str = Field(..., description="Exact phrase or minimal paraphrase of the document's core topic.")
    supporting_topics: Optional[List[str]] = Field(
        default=None,
        description="Explicitly mentioned subtopics or related areas in the document."
    )

class ShortSummary(BaseModel):
    """A concise, text-based summary limited to factual statements in the document."""
    summary_text: str = Field(..., description="Condensed version of the document’s key points (≤ 3 sentences).")

class LongSummary(BaseModel):
    """A longer, strictly factual summary based only on text evidence."""
    summary_text: str = Field(..., description="Paragraph-length summary (5–8 sentences) based solely on the document.")

class ProposedFurtherResearchTopic(BaseModel):
    """Captures explicitly stated suggestions for further research or inquiry."""
    listed_topics: Optional[List[str]] = Field(
        default=None,
        description="Sentences or bullet points directly copied or closely paraphrased from the document indicating further research areas."
    )

class KeyTerms(BaseModel):
    """Lists key terms and their definitions exactly as given in the document."""
    term: str = Field(..., description="Key term as written in the document.")
    definition: str = Field(..., description="Definition of the term as explicitly stated in the document.")

class Reference(BaseModel):
    """Represents a citation or referenced work mentioned verbatim in the text."""
    citation_text: str = Field(..., description="Exact citation line or paragraph from the document.")

# Aggregating everything together
class DocumentSummary(BaseModel):
    """Aggregates all textual summaries and metadata for one document."""
    topic: Topic
    short_summary: ShortSummary
    long_summary: LongSummary
    proposed_further_research: Optional[ProposedFurtherResearchTopic] = None
    key_terms: Optional[List[KeyTerms]] = None
    references: Optional[List[Reference]] = None

In [3]:
import os
from typing import Type
from pydantic import BaseModel
from google import genai
from google.genai import types

class DocSummarizer:
    """
    Extracts structured, strictly text-grounded summaries from Markdown documents
    into a Pydantic schema (e.g., DocumentSummary).
    No inference, guessing, or external knowledge is permitted.
    """

    def __init__(self, model: str = "gemini-2.0-flash"):
        self.model_name = model
        self.client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
        self.system_instruction = (
            "You are an expert educational summarisation agent. "
            "You analyze Markdown documents and extract ONLY what is explicitly written. "
            "Never infer, speculate, or add information. "
            "If a field is missing, leave it as null/None."
        )

    def _build_prompt(self, markdown_doc: str) -> str:
        """Builds the extraction prompt used for the summarization agent."""
        return f"""
You are an educational summarization agent.
Your task is to extract information ONLY from the given Markdown document, with zero interpretation, speculation, or external knowledge.

Follow these strict rules:
1. Use only what is explicitly written.
2. Do NOT infer or fabricate.
3. Maintain textual fidelity—copy or minimally paraphrase phrases.
4. If a field is absent, output null.
5. Output must be valid JSON conforming to the provided Pydantic schema.

DOCUMENT:
---
{markdown_doc}
---
        """.strip()

    def summariser_action(
        self,
        markdown_doc: str,
        pydantic_model: Type[BaseModel],
    ) -> BaseModel:
        """
        Summarizes a Markdown document into the provided Pydantic model.

        Args:
            markdown_doc: Raw Markdown text to analyze.
            pydantic_model: The target Pydantic BaseModel (e.g., DocumentSummary).

        Returns:
            Parsed Pydantic model with extracted content.
        """
        prompt = self._build_prompt(markdown_doc)

        config = types.GenerateContentConfig(
            system_instruction=self.system_instruction,
            response_mime_type="application/json",
            response_schema=pydantic_model,
        )

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt,
            config=config,
        )

        # Parse the JSON response into the Pydantic model
        result = pydantic_model.model_validate_json(response.text)
        print("Extraction successful!")
        return result

In [9]:
f = open('test.md', 'r') 
markdown_content = f.read()  # Store the content
f.close()

In [10]:
# API implementation

summarizer = DocSummarizer(model="gemini-2.0-flash")

# ---- Run the summarization ----
summary = summarizer.summariser_action(markdown_content, pydantic_model=DocumentSummary)

# ---- Output the parsed structure ----
print("✅ Extraction successful!\n")
print(json.dumps(summary.dict(), indent=2))

Extraction successful!
✅ Extraction successful!

{
  "topic": {
    "main_topic": "Coloured CNNs",
    "supporting_topics": [
      "Padding Techniques in Convolutional Neural Networks (CNN)",
      "Image Transformation of Images",
      "Transfer Learning"
    ]
  },
  "short_summary": {
    "summary_text": "Image classification uses filters to learn patterns. Padding adds zeros to image corners for better retention. Pooling reduces space by extracting features. Convolution and dense layers are used to classify, with learning rate, batch size, and optimizer."
  },
  "long_summary": {
    "summary_text": "Image classification uses filters passing by and learning patterns. Padding adds zeros in corners for better retention. There are multiple padding techniques including 'valid', 'same', custom, reflective, asymmetric, and circular padding. Pooling reduces space by extracting features. A flatten layer converts everything into a 1D vector. Convolution and dense layers are used to classi

/var/folders/cm/_1bs_b994bg5m57ynpdj3t780000gn/T/ipykernel_78932/3159858066.py:10: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(json.dumps(summary.dict(), indent=2))
